# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.

## 环境准备

从仓库根目录加载并检查 `.env` 与依赖配置。

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# ルートの .env を読み込む
load_dotenv()

# ルート設定を確認する
doublecheck_env("example.env")

OLLAMA_BASE_URL=<not set>
OLLAMA_MODEL=<not set>


In [2]:
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()

# Connect to the mcp-time server for timezone-aware operations
# This Go-based server provides tools for current time, relative time parsing,
# timezone conversion, duration arithmetic, and time comparison
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 5 MCP tools: ['add_time', 'compare_time', 'convert_timezone', 'current_time', 'relative_time']


Create an agent with the MCP-provided time tools.\n\nTo keep tool calls stable with local Ollama models, explicitly use valid IANA timezone names such as `America/Los_Angeles`.


In [3]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="ollama:qwen2.5:7b",
    tools=mcp_tools,
    system_prompt=(
        "You are a helpful assistant. When using time tools, always pass valid IANA timezone names. "
        "For San Francisco, use America/Los_Angeles."
    ),
)


Ask about the current time in San Francisco using its explicit IANA timezone.


In [4]:
result = await agent_with_mcp.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the current time in America/Los_Angeles? This is the timezone for San Francisco.",
            }
        ]
    }
)
for msg in result["messages"]:
    msg.pretty_print()


================================ Human Message =================================

What's the current time in America/Los_Angeles? This is the timezone for San Francisco.
================================== Ai Message ==================================
Tool Calls:
  current_time (714b2df9-8751-403f-b6cf-e68b7cd6176e)
 Call ID: 714b2df9-8751-403f-b6cf-e68b7cd6176e
  Args:
    format: DateTime
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': '2026-03-07 01:40:16', 'id': 'lc_61b267fa-255d-4a14-b912-2073774effc2'}]
================================== Ai Message ==================================

The current time in America/Los_Angeles (San Francisco) is 01:40:16 on March 7, 2026.
